# 리포트 42 — 두 기체가 갈리는 이유는 메쉬 품질이 아니라 표적 크기 대비 광선예산이다

> ### 한 일
> **기하를 고정하고 광선예산만 16배 흔들어 다시 추적해 판정 통계가 경로수를 어떻게 따라가는지 쟀다.**

### 결과
1. 판정 통계는 경로수 1 dB 당 1.01 dB [^1] 로 따라 올라간다 — 칸 6 개 [^2] 의 중앙값이다.
2. 두 기체의 프롭 경로수 비는 5.41 배 [^3] (Matrice 4E / Mini 2) 다. 예산 보정을 하면 자세 짝 15 [^4] 중 12 [^5] 에서 Mini 2 가 앞선다.
3. 예산 보정 차이의 중앙값은 -5.73 dB [^6] 이고, 경로수를 맞추면 Mini 2 가 +7.40 dB [^7] 올라간다.
4. ⚠ 예산을 맞추려면 5.41 배 [^8] 가 필요한데 표본 인자가 uint32 라 올릴 수 있는 여유는 2.0 배 [^9] 다 — 이 하네스로 기체 비교를 닫으려면 표본 규칙을 먼저 바꿔야 한다.
5. 그래서 Mini 2 의 무변조 칸 10/oblique/prod/prop [^10] 계열은 표적의 성질이 아니라 예산의 성질이다.

### 방법

| 무엇을 | 어떻게 얻었나 |
|---|---|
| 예산 사다리 | 기하·자세·재질을 고정하고 표본 인자만 16 배 범위로 올려 새 시드로 다시 추적했다 |
| 기울기 | 판정 통계 [dB] 를 평균 경로수 [dB] 에 회귀한 기울기. 1 에 붙으면 통계가 신호 세기가 아니라 **확신도**를 재고 있다는 뜻이다 |
| 기체 짝 보정 | 두 기체의 같은 칸에서 판정 통계 차이를 경로수 차이로 나눠 보정했다 |
| 독립 재현 | 같은 표본 인자·같은 칸을 새 시드 집합으로 다시 추적해 본 격자 값과 맞댔다 |
| 모양 불변성 | 코히런트 조화의 **모양**이 예산에 불변한 거리를 따로 쟀다 — 그 거리 안에서만 상대 스펙트럼을 인용한다 |

### 재현

```bash
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15_attack_spp_ladder.py
PYTHONPATH=src ~/.venvs/py312/bin/python benchmark/report15_attack_stats.py
```

| | |
|---|---|
| 출력 | `outputs/report15_attack_spp_ladder.json`, `outputs/report15_attack_stats.json` |
| 소요 | 약 36 분 (GPU 1장 — 사다리 전량 재추적) |

### 앞 편에서

| 어디서 | 무엇을 알고 와야 하나 |
|---|---|
| 앞 편 | [편 41 «판정 잣대 교정»](41_md-calibration.ipynb) — 이 통계의 문턱 |

---

## 판정량이 예산에 1 대 1 로 매달려 있다

![report15_f3](../outputs/figures/report15_f3.png)

**그림 1.** 경로수가 위상마다 껐다 켜지는가, 그리고 두 엔진은 거리에 따라 어떻게 갈리는가?
위 두 칸이 확산 채널의 경로수다 — 위상에 따라 매끄럽게 변하고, 껐다 켜지는 칸은 0 개다. 가운데가 정반사 채널이고, 아래가 거리별 두 엔진 일치도다.

오른쪽 아래에서 Matrice 4E 의 «hot» 자세만 거리와 함께 무너진다 — 그 자세의 경로수가 가장 적기 때문이다.

## 예산 법칙

| 무엇을 | 값 |
|---|---|
| 경로수 1 dB 당 판정 통계 상승 | 1.01 dB [^1] |
| 잰 칸 수 | 6 개 [^2] |
| 두 기체 경로수 비 | 5.41 배 [^3] |
| 예산 보정 차이(중앙값) | -5.73 dB [^6] |
| 보정 뒤 Mini 2 가 앞서는 짝 | 12 [^5] / 15 [^4] |
| 경로수를 맞췄을 때 Mini 2 상승분 | +7.40 dB [^7] |

기울기가 1 에 붙는다는 것은 이 통계가 **신호 세기가 아니라 확신도**를 잰다는 뜻이다. 그러므로 경로수가 다른 두 대상끼리 이 값을 직접 견주는 길은 막혀 있다.

## 하네스의 표본 상한이 그 비교를 막는다

| 무엇을 | 값 |
|---|---|
| 예산을 맞추는 데 필요한 배수 | 5.41 배 [^8] |
| 하네스가 올릴 수 있는 배수 | 2.0 배 [^9] |
| 본 격자 표본 인자 | 2,048,000,000 [^11] |
| 안전 상한 | 4,096,000,000 [^12] |
| 맞출 수 있나 | 아니오 [^13] |

표본 인자가 uint32 라 상한이 있다. 그래서 «Mini 2 가 Matrice 4E 보다 못하다» 는 읽기는 이 하네스 안에서 근거를 잃는다 — 예산 법칙을 그대로 적용하면 Mini 2 의 무변조 칸이 전부 사라진다는 것이 그 예측이고, 그 예측을 확인하려면 표본 상한을 먼저 올려야 한다.

## 꼬리도 같은 이유에서 나온다

| 기체 | Sionna 꼬리(중앙값) | PO 꼬리 | −20 dB 위 칸 수 |
|---|---|---|---|
| Mini 2 | -19.56 dB [^14] | -36.14 dB [^15] | 8 [^16] / 15 [^17] |
| Matrice 4E | -24.60 dB [^18] | -35.83 dB [^19] | 3 [^20] / 15 [^21] |

프롭 경로가 적은 기체에서 꼬리가 더 높이 남고, 그 꼬리가 −20 dB 가장자리 판정을 오염시킨다. Mini 2 는 Das 실측으로 검증된 기준자이므로, 이 차이를 «메쉬 탓» 으로 읽는 것은 근거를 벗어난다.

## 어느 거리에서 무엇을 인용해도 되나

| 거리 [m] | 코히런트 모양 코사인 | 최저 예산에서의 경로수 | 모양이 예산에 불변인가 |
|---|---|---|---|
| 1 | 0.9848 | 20,329 | 예 |
| 3 | 0.9373 | 2,349 | 아니오 |
| 10 | 0.7303 | 202 | 아니오 |

출처 [^22]

가까운 거리에서는 상대 도플러 스펙트럼을 읽어도 된다. 거리가 늘면 모양 자체가 예산을 따라 흔들리므로 상대 스펙트럼도 인용 범위 밖이다. **절대 레벨은 어느 거리에서도 인용하지 않는다** — 레벨이 √N 으로 자라기 때문이다.

## 다음 단계

| 다음에 할 일 | 그러면 결정되는 것 | 어디서 |
|---|---|---|
| 경로수를 표적 크기에 맞춰 정규화하는 예산 규칙을 하네스에 넣는다 | 기체 비교가 «표적의 성질» 로 닫힌다 — 지금은 예산의 성질이 섞여 있다 | `benchmark/report15_probe.py` 의 표본 인자 규칙 |
| 가장자리 판정을 꼬리 초과분으로 보정한다 | «가장자리가 예측과 맞는가» 가 꼬리와 무관해진다 | [편 36 «두 엔진»](36_md-two-engines.ipynb) |
| 모양 불변 거리 밖의 칸을 인용 목록에서 뺀다 | 상대 스펙트럼 인용의 성립 범위가 거리로 못 박힌다 | 이 편의 거리 표 |

<!--rs:sources-->
## 출처

본문의 `[^n]` 은 아래 22개 중 하나를 가리킨다. 값은 이 표를 만들 때 JSON 을 다시 열어 채웠다 — 본문 숫자와 같은 파일, 같은 키다.

| | 파일 | 키 | 값 |
|---|---|---|---|
| [^1] | `outputs/report15_attack_stats.json` | `q1_noise_floor.budget_law.slope_db_per_db_of_paths_median` | 1.01 |
| [^2] | `outputs/report15_attack_stats.json` | `q1_noise_floor.budget_law.slope_n_cells` | 6 |
| [^3] | `outputs/report15_attack_stats.json` | `q1_noise_floor.budget_law.median_path_count_ratio_m4e_over_mini2` | 5.409 |
| [^4] | `outputs/report15_attack_stats.json` | `q1_noise_floor.budget_law.n_pairs` | 15 |
| [^5] | `outputs/report15_attack_stats.json` | `q1_noise_floor.budget_law.n_pairs_where_mini2_wins_after_correction` | 12 |
| [^6] | `outputs/report15_attack_stats.json` | `q1_noise_floor.budget_law.budget_corrected_delta_median_db` | -5.735 |
| [^7] | `outputs/report15_attack_stats.json` | `q1_noise_floor.budget_law.implied_mini2_boost_if_budget_matched_db` | 7.402 |
| [^8] | `outputs/report15_attack_stats.json` | `q1_noise_floor.budget_law.spp_cap.factor_needed_for_match` | 5.409 |
| [^9] | `outputs/report15_attack_stats.json` | `q1_noise_floor.budget_law.spp_cap.factor_available` | 2 |
| [^10] | `outputs/report15_attack_stats.json` | `q1_noise_floor.budget_law.mini2_no_modulation_cells_now[0]` | 10/oblique/prod/prop |
| [^11] | `outputs/report15_attack_stats.json` | `q1_noise_floor.budget_law.spp_cap.used_in_grid` | 2048000000 |
| [^12] | `outputs/report15_attack_stats.json` | `q1_noise_floor.budget_law.spp_cap.max_safe_in_harness` | 4096000000 |
| [^13] | `outputs/report15_attack_stats.json` | `q1_noise_floor.budget_law.spp_cap.match_is_possible` | 아니오 |
| [^14] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.mini2.sionna_tail_max_db_median` | -19.56 |
| [^15] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.mini2.po_tail_max_db_median` | -36.14 |
| [^16] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.mini2.n_cells_sionna_tail_above_minus20` | 8 |
| [^17] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.mini2.n_cells` | 15 |
| [^18] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.matrice4e.sionna_tail_max_db_median` | -24.6 |
| [^19] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.matrice4e.po_tail_max_db_median` | -35.83 |
| [^20] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.matrice4e.n_cells_sionna_tail_above_minus20` | 3 |
| [^21] | `outputs/report15_verdict.json` | `tail_excess.by_airframe.matrice4e.n_cells` | 15 |
| [^22] | `outputs/report15_attack_stats.json` | `q1_noise_floor.spp_dependence_existing.shape_invariance.by_range` | (3항목 묶음) |